# Smart Traffic Control System - Model Training

In this notebook we train 3 models to detect vehicles and compare them.

1. YOLO11
2. Faster R-CNN
3. RetinaNet

We also train a small model that tells if a frame has an accident or not.

Before running: Runtime > Change runtime type > T4 GPU

## 1. Install and import

In [ ]:
!pip install -q ultralytics roboflow kagglehub torchmetrics pycocotools

In [ ]:
import os, glob, time, shutil, yaml
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

CLASSES = ["car", "motorcycle", "bus", "truck"]
CONF = 0.4                      # boxes above this score are counted as vehicles
results = {}                    # scores of every model
loss_history = {}               # training loss of every model
os.makedirs("models", exist_ok=True)

## 2. Dataset

We use 2 datasets from Kaggle and 2 from Roboflow.
All of them have different class names, so we rename them to our 4 classes and put everything in one folder called `data`.

In [ ]:
# download from Kaggle
import kagglehub
sources = {}
kaggle_names = {
    "traffic":  "yusufberksardoan/traffic-detection-project",
    "vehicles": "saumyapatel/traffic-vehicles-object-detection",
}
for tag, name in kaggle_names.items():
    try:
        sources[tag] = kagglehub.dataset_download(name)
    except Exception as e:
        print(name, "failed:", e)

# download from Roboflow
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY")       # paste your key here

def get_dataset(project, versions):
    # try each version and keep the first one that really has images
    for v in versions:
        try:
            folder = rf.workspace("traffic").project(project).version(v).download("yolov8").location
            if glob.glob(f"{folder}/**/*.jpg", recursive=True):
                return folder
        except Exception as e:
            print(project, "version", v, "failed:", e)
    print(project, ": no usable version found")

sources["roboflow1"] = get_dataset("traffic-dataset-z21ak", [1, 2])
sources["roboflow2"] = get_dataset("2600dataset", [1])
sources

In [ ]:
# names used in the datasets -> our class names
RENAME = {
    "car": "car", "cars": "car", "van": "car", "ambulance": "car",
    "motorcycle": "motorcycle", "motocycle": "motorcycle", "motorbike": "motorcycle", "bike": "motorcycle", "two wheeler": "motorcycle",
    "bus": "bus", "truck": "truck", "trucks": "truck",
}
MAX_IMAGES = 1500       # images taken from each dataset

def add_dataset(folder, tag):
    yaml_files = glob.glob(f"{folder}/**/data.yaml", recursive=True)
    if not yaml_files:
        print(tag, ": data.yaml not found, skipped")
        return
    names = yaml.safe_load(open(yaml_files[0]))["names"]
    images = sorted(glob.glob(f"{folder}/**/images/*.*", recursive=True))
    has_valid = any("/valid/" in img for img in images)
    count = 0

    for img in images:
        label_file = os.path.splitext(img.replace("/images/", "/labels/"))[0] + ".txt"
        if count >= MAX_IMAGES or "/test/" in img or not os.path.exists(label_file):
            continue

        # keep only our 4 classes and write the new class number
        lines = []
        for line in open(label_file):
            p = line.split()
            if len(p) < 5:
                continue
            name = str(names[int(p[0])]).lower().replace("_", " ")
            if name not in RENAME:
                continue
            v = [float(a) for a in p[1:]]
            if len(v) > 4:          # label is an outline, change it to a box
                xs, ys = v[0::2], v[1::2]
                v = [(min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2, max(xs) - min(xs), max(ys) - min(ys)]
            lines.append(f"{CLASSES.index(RENAME[name])} {v[0]} {v[1]} {v[2]} {v[3]}")
        if len(lines) == 0:
            continue

        # use the valid folder of the dataset, or every 5th image if it has none
        if has_valid:
            split = "valid" if "/valid/" in img else "train"
        else:
            split = "valid" if count % 5 == 0 else "train"
        shutil.copy(img, f"data/{split}/images/{tag}_{count}{os.path.splitext(img)[1]}")
        open(f"data/{split}/labels/{tag}_{count}.txt", "w").write("\n".join(lines))
        count += 1
    print(tag, ":", count, "images added")

shutil.rmtree("data", ignore_errors=True)
for split in ["train", "valid"]:
    os.makedirs(f"data/{split}/images")
    os.makedirs(f"data/{split}/labels")

for tag, folder in sources.items():
    if folder:
        add_dataset(folder, tag)

data_yaml = {"path": os.path.abspath("data"), "train": "train/images", "val": "valid/images", "names": dict(enumerate(CLASSES))}
yaml.safe_dump(data_yaml, open("data/data.yaml", "w"))

train_images = sorted(glob.glob("data/train/images/*"))
valid_images = sorted(glob.glob("data/valid/images/*"))
print("train:", len(train_images), "valid:", len(valid_images))

In [ ]:
# how many boxes of each class we have for training
box_count = [0, 0, 0, 0]
for f in glob.glob("data/train/labels/*.txt"):
    for line in open(f):
        box_count[int(line.split()[0])] += 1

sns.barplot(x=CLASSES, y=box_count)
plt.title("Boxes per class in training data")
plt.show()

## 3. Common functions

These functions are used by all 3 models, so every model is tested in the same way.

In [ ]:
from torchvision.transforms.functional import to_tensor
from torchvision.ops import box_iou
from torchmetrics.detection import MeanAveragePrecision
from sklearn.metrics import confusion_matrix, r2_score, mean_absolute_error

# read the real boxes of one image (YOLO text file -> pixel boxes)
def read_labels(img_path):
    w, h = Image.open(img_path).size
    label_file = os.path.splitext(img_path.replace("/images/", "/labels/"))[0] + ".txt"
    boxes, labels = [], []
    for line in open(label_file):
        c, x, y, bw, bh = map(float, line.split())
        if bw * w > 1 and bh * h > 1:
            boxes.append([(x - bw / 2) * w, (y - bh / 2) * h, (x + bw / 2) * w, (y + bh / 2) * h])
            labels.append(int(c))
    return torch.tensor(boxes).reshape(-1, 4), torch.tensor(labels, dtype=torch.int64)

# get boxes, labels and scores from a model for one image
def predict(name, model, img_path):
    if name == "YOLO11":
        r = model(img_path, conf=0.05, verbose=False)[0]
        return r.boxes.xyxy.cpu(), r.boxes.cls.cpu().long(), r.boxes.conf.cpu()
    img = to_tensor(Image.open(img_path).convert("RGB")).to(device)
    with torch.no_grad():
        out = model([img])[0]
    return out["boxes"].cpu(), out["labels"].cpu() - 1, out["scores"].cpu()     # -1 because 0 is background

In [ ]:
# test a model on the validation images and save its scores
def evaluate(name, model, model_file):
    if name != "YOLO11":
        model.eval()
    metric = MeanAveragePrecision()
    y_true, y_pred = [], []
    true_count, pred_count = [], []
    correct, seconds = 0, 0

    for img_path in valid_images:
        start = time.time()
        boxes, labels, scores = predict(name, model, img_path)
        seconds += time.time() - start
        real_boxes, real_labels = read_labels(img_path)
        metric.update([{"boxes": boxes, "labels": labels, "scores": scores}], [{"boxes": real_boxes, "labels": real_labels}])

        # keep confident boxes and match them with the real boxes (IoU >= 0.5)
        boxes, labels = boxes[scores >= CONF], labels[scores >= CONF]
        true_count.append(len(real_boxes))
        pred_count.append(len(boxes))
        iou = box_iou(real_boxes, boxes)
        used = []
        for t in range(len(real_boxes)):
            best = -1
            for p in range(len(boxes)):
                if p not in used and iou[t, p] >= 0.5 and (best == -1 or iou[t, p] > iou[t, best]):
                    best = p
            y_true.append(CLASSES[real_labels[t]])
            if best == -1:
                y_pred.append("missed")
            else:
                used.append(best)
                y_pred.append(CLASSES[labels[best]])
                if labels[best] == real_labels[t]:
                    correct += 1

    m = metric.compute()
    precision = correct / max(1, sum(pred_count))
    recall = correct / max(1, sum(true_count))
    results[name] = {
        "Precision": precision,
        "Recall": recall,
        "F1": 2 * precision * recall / max(precision + recall, 1e-9),
        "mAP50": m["map_50"].item(),
        "mAP50-95": m["map"].item(),
        "Count R2": r2_score(true_count, pred_count),
        "Count MAE": mean_absolute_error(true_count, pred_count),
        "FPS": len(valid_images) / seconds,
        "Size MB": os.path.getsize(model_file) / 1e6,
    }
    print(name, results[name])

    # confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=CLASSES + ["missed"])
    sns.heatmap(cm[:4], annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES + ["missed"], yticklabels=CLASSES)
    plt.xlabel("Predicted"); plt.ylabel("Real"); plt.title(name + " confusion matrix")
    plt.show()

In [ ]:
# data loader and training loop for Faster R-CNN and RetinaNet
class VehicleData(torch.utils.data.Dataset):
    def __len__(self):
        return len(train_images)

    def __getitem__(self, i):
        img = to_tensor(Image.open(train_images[i]).convert("RGB"))
        boxes, labels = read_labels(train_images[i])
        return img, {"boxes": boxes, "labels": labels + 1}       # +1 because 0 is background

def collate(batch):
    return tuple(zip(*batch))

train_loader = torch.utils.data.DataLoader(VehicleData(), batch_size=4, shuffle=True, collate_fn=collate, num_workers=2)

def train(name, model, epochs):
    model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0001)
    loss_history[name] = []
    for epoch in range(epochs):
        model.train()
        total = 0
        for images, targets in train_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss = sum(model(images, targets).values())
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 10)
            optimizer.step()
            total += loss.item()
        loss_history[name].append(total / len(train_loader))
        print("epoch", epoch + 1, "loss", round(loss_history[name][-1], 3))

## 4. Model 1 - YOLO11

YOLO looks at the image only once and gives all the boxes together, so it is fast.

In [ ]:
from ultralytics import YOLO

yolo = YOLO("yolo11s.pt")
yolo.train(data="data/data.yaml", epochs=30, imgsz=640, name="yolo11")
yolo_dir = str(yolo.trainer.save_dir)
shutil.copy(yolo_dir + "/weights/best.pt", "models/yolo11.pt")

In [ ]:
# training loss of YOLO in every epoch
log = pd.read_csv(yolo_dir + "/results.csv")
log.columns = log.columns.str.strip()
loss_history["YOLO11"] = list(log["train/box_loss"] + log["train/cls_loss"] + log["train/dfl_loss"])

yolo = YOLO("models/yolo11.pt")
evaluate("YOLO11", yolo, "models/yolo11.pt")

In [ ]:
# some predictions of YOLO
r = yolo(valid_images[0], conf=CONF, verbose=False)[0]
plt.imshow(r.plot()[:, :, ::-1]); plt.axis("off"); plt.title("YOLO11 prediction")
plt.show()

## 5. Model 2 - Faster R-CNN

Faster R-CNN works in 2 steps. First it finds areas where an object can be, then it classifies each area.
It is accurate but slower than YOLO.

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# pretrained model, we only change the last layer to our 4 classes + background
frcnn = fasterrcnn_resnet50_fpn(weights="DEFAULT", min_size=640, max_size=640)
in_features = frcnn.roi_heads.box_predictor.cls_score.in_features
frcnn.roi_heads.box_predictor = FastRCNNPredictor(in_features, 5)

train("Faster R-CNN", frcnn, epochs=8)
torch.save(frcnn.state_dict(), "models/fasterrcnn.pth")
evaluate("Faster R-CNN", frcnn, "models/fasterrcnn.pth")

## 6. Model 3 - RetinaNet

RetinaNet is a one step model like YOLO. It uses focal loss, which makes the model focus on the difficult objects.

In [ ]:
from torchvision.models.detection import retinanet_resnet50_fpn
from torchvision.models.detection.retinanet import RetinaNetClassificationHead

# pretrained model, we replace the part that predicts the class
retina = retinanet_resnet50_fpn(weights="DEFAULT", min_size=640, max_size=640)
num_anchors = retina.head.classification_head.num_anchors
retina.head.classification_head = RetinaNetClassificationHead(256, num_anchors, 5)

train("RetinaNet", retina, epochs=8)
torch.save(retina.state_dict(), "models/retinanet.pth")
evaluate("RetinaNet", retina, "models/retinanet.pth")

## 7. Accident model

This model looks at the full frame and says accident or no accident.
Dataset: Accident Detection From CCTV Footage (Kaggle)

In [ ]:
acc_folder = kagglehub.dataset_download("ckay16/accident-detection-from-cctv-footage")
for root, dirs, files in os.walk(acc_folder):       # find the folder that has train / val inside
    if "train" in dirs:
        acc_data = root
print(acc_data, os.listdir(acc_data))

acc = YOLO("yolo11n-cls.pt")
acc.train(data=acc_data, epochs=15, imgsz=224, name="accident")
acc_dir = str(acc.trainer.save_dir)
shutil.copy(acc_dir + "/weights/best.pt", "models/accident.pt")

acc_val = YOLO("models/accident.pt").val(data=acc_data)
print("Accident model accuracy:", round(acc_val.top1, 3))
display(Image.open(str(acc_val.save_dir) + "/confusion_matrix.png").resize((600, 450)))

## 8. Comparison of the 3 models

- Precision: out of the boxes the model made, how many are correct
- Recall: out of the real vehicles, how many the model found
- F1: balance of precision and recall
- mAP50 and mAP50-95: main accuracy score of detection models
- Count R2 and Count MAE: how close the vehicle count of the model is to the real count
- FPS: images per second (speed)

In [ ]:
table = pd.DataFrame(results).T.round(3)
table.to_csv("models/comparison.csv")
display(table)
print("Best model:", table["mAP50-95"].idxmax())

In [ ]:
# bar graph of the accuracy scores
scores = table[["Precision", "Recall", "F1", "mAP50", "mAP50-95"]].reset_index().melt(id_vars="index")
scores.columns = ["Model", "Metric", "Score"]

plt.figure(figsize=(10, 5))
sns.barplot(data=scores, x="Metric", y="Score", hue="Model")
plt.title("Accuracy of the 3 models")
plt.ylim(0, 1)
plt.show()

# bar graph of speed and counting error
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(x=table.index, y=table["FPS"], ax=ax[0])
ax[0].set_title("Speed (FPS, higher is better)")
sns.barplot(x=table.index, y=table["Count MAE"], ax=ax[1])
ax[1].set_title("Counting error (MAE, lower is better)")
plt.show()

In [ ]:
# line graph of training loss
# the loss of each model is calculated in a different way, so every model has its own graph
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for i, name in enumerate(loss_history):
    epochs = range(1, len(loss_history[name]) + 1)
    sns.lineplot(x=epochs, y=loss_history[name], marker="o", ax=ax[i])
    ax[i].set_title(name + " training loss")
    ax[i].set_xlabel("Epoch")
    ax[i].set_ylabel("Loss")
plt.show()

## 9. Download the trained models

In [ ]:
from google.colab import files
shutil.make_archive("models", "zip", "models")
files.download("models.zip")